<a href="https://colab.research.google.com/github/yianli0213/programming-language/blob/main/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q google-generativeai

In [5]:
import gradio as gr
import pandas as pd
import gspread
from google.colab import auth, userdata
from google.auth import default
from datetime import datetime
from google import genai

# 從 Colab Secrets 取得 Gemini API 金鑰
api_key = userdata.get('gemini')
client = genai.Client(api_key=api_key)
MODEL_ID = 'gemini-2.5-flash'

In [6]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.


In [7]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1GPw6CWyekRaEt1hJ4F6mnW4ucXs8XGuptyId3B78txc/edit?usp=sharing"
COLUMNS = ["日期", "科目", "章節", "考試類型", "分數", "滿分"]

_gc = None
_sh = None

def ensure_connection():
    global _gc, _sh
    if _gc is None:
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _sh = _gc.open_by_url(SHEET_URL)

ensure_connection()

In [8]:
def get_or_create_student_ws(name):
    try:
        ws = _sh.worksheet(name)
    except gspread.WorksheetNotFound:
        ws = _sh.add_worksheet(title=name, rows=1000, cols=10)
        ws.append_row(COLUMNS)
    return ws

def list_students():
    return [ws.title for ws in _sh.worksheets() if ws.title != '工作表1']

def get_student_df(name):
    ws = get_or_create_student_ws(name)
    data = ws.get_all_values()
    if len(data) <= 1:
        return pd.DataFrame(columns=COLUMNS)
    return pd.DataFrame(data[1:], columns=data[0])

In [9]:
def add_grade(student, subject, chapter, exam_type, score, total):
    if not student:
        return '⚠️ 請先選擇或輸入學生名字', None
    ws = get_or_create_student_ws(student)
    today = datetime.now().strftime('%Y-%m-%d')
    ws.append_row([today, subject, chapter, exam_type, float(score), float(total)])
    df = get_student_df(student)
    return f'✅ 已新增 {student} 的成績（{subject} / {chapter}）', df

def load_student_grades(student):
    if not student:
        return pd.DataFrame(columns=COLUMNS)
    return get_student_df(student)

In [10]:
def analyze_student(student):
    if not student:
        return '⚠️ 請先選擇學生'

    df = get_student_df(student)
    if df.empty:
        return '這位學生還沒有成績資料，請先新增成績。'

    df['分數'] = pd.to_numeric(df['分數'], errors='coerce')
    df['滿分'] = pd.to_numeric(df['滿分'], errors='coerce')
    df['答對率'] = df['分數'] / df['滿分'] * 100

    summary = df.groupby(['科目', '章節'])['答對率'].mean().reset_index()
    summary.columns = ['科目', '章節', '平均答對率']
    summary['平均答對率'] = summary['平均答對率'].round(1)

    prompt = f"""
以下是我的學生「{student}」各科目章節的歷史成績（平均答對率 %）：

{summary.to_string(index=False)}

請你：
1. 找出平均答對率低於 70% 的章節，標記為「需要加強」
2. 找出平均答對率高於 85% 的章節，標記為「表現穩定」
3. 針對最弱的 2~3 個章節，給出具體的複習方向建議
4. 最後用 2~3 句話總結這位學生整體的學習狀況

請用繁體中文回答，條列清楚。
"""

    try:
        response = client.models.generate_content(model=MODEL_ID, contents=prompt)
        result = response.text

        # 寫入該學生的 Google Sheet
        ws = get_or_create_student_ws(student)
        today = datetime.now().strftime('%Y-%m-%d')
        ws.append_row([today, 'AI總結', '', '', '', result])

        return result
    except Exception as e:
        return f'AI 分析失敗：{e}'

In [11]:
def refresh_students():
    return gr.Dropdown(choices=list_students())

with gr.Blocks(title='家教成績管理系統') as demo:
    gr.Markdown('# 📚 家教成績管理系統')

    with gr.Tab('➕ 新增成績'):
        with gr.Row():
            student_input = gr.Dropdown(
                choices=list_students(),
                label='選擇學生',
                allow_custom_value=True,
                info='選擇現有學生，或直接輸入新學生名字'
            )
            btn_refresh = gr.Button('🔄 更新學生清單')

        with gr.Row():
            subject = gr.Dropdown(
                ['數學', '國文', '英文', '理化', '生物', '歷史', '地理', '公民'],
                label='科目'
            )
            chapter = gr.Textbox(label='章節', placeholder='例如：一元一次方程式')
            exam_type = gr.Dropdown(['段考', '小考', '模擬題', '會考'], label='考試類型')

        with gr.Row():
            score = gr.Number(label='分數', value=0)
            total = gr.Number(label='滿分', value=100)

        btn_add = gr.Button('➕ 新增成績', variant='primary')
        msg = gr.Textbox(label='狀態', interactive=False)
        grade_table = gr.Dataframe(label='目前成績紀錄', interactive=False)

    with gr.Tab('🧠 AI 弱點分析'):
        with gr.Row():
            student_analyze = gr.Dropdown(choices=list_students(), label='選擇學生')
            btn_analyze_refresh = gr.Button('🔄 更新學生清單')
        btn_analyze = gr.Button('開始分析弱點', variant='primary')
        analysis_output = gr.Textbox(label='AI 分析結果', lines=20, interactive=False)

    # 綁定按鈕
    btn_refresh.click(fn=refresh_students, outputs=student_input)
    btn_analyze_refresh.click(fn=refresh_students, outputs=student_analyze)
    student_input.change(fn=load_student_grades, inputs=student_input, outputs=grade_table)
    btn_add.click(
        fn=add_grade,
        inputs=[student_input, subject, chapter, exam_type, score, total],
        outputs=[msg, grade_table]
    )
    btn_analyze.click(
        fn=analyze_student,
        inputs=student_analyze,
        outputs=analysis_output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://942ddf6c6040b4b900.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
